## Configure API Authentication, Storage Credentials and Analyzer Schema

In [1]:
import os
import requests
import json
import uuid

# Update with your AI Services Resource Credentials
os.environ['ENDPOINT'] = ""
os.environ['RESOURCE_KEY'] = ""

#Content Understanding API Version, replace with your target API version
os.environ['API_VERSION'] = "2024-12-01-preview"

#Set up Analyzer schema and Analyzer ID. (This is an example schema, replace it with your actual content)
os.environ['REQUEST_BODY_WITH_ANALYZER_SCHEMA'] = json.dumps({
    "description": "Sample invoice analyzer",
    "scenario": "document",
    "config": {
        "returnDetails": True
    },
    "fieldSchema": {
        "fields": {
            "VendorName": {
                "type": "string",
                "method": "extract",
                "description": "Vendor issuing the invoice"
            },
            "Items": {
                "type": "array",
                "method": "extract",
                "items": {
                    "type": "object",
                    "properties": {
                        "Description": {
                            "type": "string",
                            "method": "extract",
                            "description": "Description of the item"
                        },
                        "Amount": {
                            "type": "number",
                            "method": "extract",
                            "description": "Amount of the item"
                        }
                    }
                }
            }
        }
    }
})

#You can replace "sample_invoice" below with a string of your choosing
os.environ['ANALYZER_ID'] = "sample_invoice" + str(uuid.uuid4())

# Update with your Blob Storage credentials and container names
os.environ['STORAGE_ACCOUNT_NAME'] = ""
os.environ['STORAGE_ACCOUNT_SAS_TOKEN'] = ""
os.environ['INPUT_CONTAINER'] = ""
os.environ['OUTPUT_CONTAINER'] = ""



## Create A New Content Understanding Analyzer or Update Existing One

In [ ]:
import os
import requests
import json

#Retrieve Environment Variables
endpoint = os.getenv('ENDPOINT')
analyzer_id = os.getenv('ANALYZER_ID')
api_version = os.getenv('API_VERSION')
resource_key = os.getenv('RESOURCE_KEY')
request_body = json.loads(os.getenv('REQUEST_BODY_WITH_ANALYZER_SCHEMA'))

# Construct the URL for the PUT request to create or update your content understanding analyzer
put_url = f"{endpoint}/contentunderstanding/analyzers/{analyzer_id}?api-version={api_version}"


# Define the headers for the PUT request
headers = {
    "Ocp-Apim-Subscription-Key": resource_key,
    "Content-Type": "application/json"
}

# Perform the PUT request to create/update the analyzer
put_response = requests.put(put_url, headers=headers, json=request_body)


#Save Operation Location
operation_location = put_response.headers.get('Operation-Location')

## Check Analyzer Creation Status

In [ ]:
# Construct the URL for the GET request to check operation status for Analyzer Creation
get_url = str(operation_location)

# Define the headers for the GET request
get_headers = {
    "Ocp-Apim-Subscription-Key": resource_key
}

# Perform the GET request to check the operation status and print
get_response = requests.get(get_url, headers=get_headers)

response = get_response.json()

status_code = response.get("status")

#Print request status, "Succeded" is expected if the request was successful
print("Status:", status_code)

## Analyze Files and Upload Results to Blob Storage Container

In [ ]:
import os
import json
import time
import requests
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient, ContainerClient
from urllib.parse import urljoin

# Retrieve environment variables
storage_account_name = os.getenv('STORAGE_ACCOUNT_NAME')
storage_account_sas_token = os.getenv('STORAGE_ACCOUNT_SAS_TOKEN')
input_container = os.getenv('INPUT_CONTAINER')
output_container = os.getenv('OUTPUT_CONTAINER')
resource_key = os.getenv('RESOURCE_KEY')
endpoint = os.getenv('ENDPOINT')
analyzer_id = os.getenv('ANALYZER_ID')
api_version = "2024-12-01-preview"  # Make sure this matches your deployed model

# Setup blob clients
account_url = f"https://{storage_account_name}.blob.core.windows.net"
credential = DefaultAzureCredential()
blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)

# Input and output container clients using SAS tokens
input_container_url = f"https://{storage_account_name}.blob.core.windows.net/{input_container}?{storage_account_sas_token}"
input_container_client = ContainerClient.from_container_url(input_container_url)
output_container_url = f"https://{storage_account_name}.blob.core.windows.net/{output_container}?{storage_account_sas_token}"
output_container_client = ContainerClient.from_container_url(output_container_url)

# List blobs in input container
input_blobs = input_container_client.list_blobs()

# Function to upload blobs to output container
def upload_blob_to_container(blob_service_client, container_url, blob_name, content):
    container_client = ContainerClient.from_container_url(container_url)
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(content, overwrite=True)

# Polling function to wait for analysis results
def wait_for_analysis_completion(result_url, headers, timeout=120, poll_interval=5):
    start_time = time.time()
    while True:
        response = requests.get(result_url, headers=headers)
        response.raise_for_status()
        result = response.json()
        status = result.get("status")
        print(f"Status: {status}")

        if status == "Succeeded":
            return result
        elif status == "Failed":
            raise RuntimeError("Analysis failed.")

        if time.time() - start_time > timeout:
            raise TimeoutError("Timed out waiting for analysis to complete.")

        time.sleep(poll_interval)

# Main processing loop for all files
headers = {
    "Ocp-Apim-Subscription-Key": resource_key,
    "Content-Type": "application/json"
}

for blob in input_blobs:
    file_name = blob.name

    if file_name.endswith('.json'):
        continue

    file_url = f"https://{storage_account_name}.blob.core.windows.net/{input_container}/{file_name}?{storage_account_sas_token}"
    print(f"\nProcessing file: {file_name}")

    r = requests.get(file_url)

    result = {
        "status_code": r.status_code,
        "blob_access": "Success" if r.status_code == 200 else "Failed",
        "response_preview": r.text[:500]  # Trim long content
    }

    if r.status_code != 200:
        print(f"Skipping {file_name} due to failed access.")
        continue

    # Step 1: Submit analyze request with retry logic
    analyze_url = f"{endpoint}contentunderstanding/analyzers/{analyzer_id}:analyze?api-version={api_version}"
    payload = {"url": str(file_url)}

    max_retries = 3
    for attempt in range(max_retries):
        try:
            analyze_response = requests.post(analyze_url, headers=headers, json=payload)
            analyze_response.raise_for_status()
            break  # Success
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                raise Exception(
                    f"Final attempt failed. Payload: {payload}, Headers: {headers}, URL: {analyze_url}"
                )
            time.sleep(2 * (attempt + 1))  # Exponential backoff

    # Step 2: Wait for result to be ready
    operation_location = analyze_response.headers.get("operation-location")
    if not operation_location:
        raise ValueError("No operation-location returned in analyze response")

    result = wait_for_analysis_completion(operation_location, headers)

    # Step 3: Upload result JSON to blob named after input file
    output_blob_name = os.path.splitext(file_name)[0] + "_result.json"
    upload_blob_to_container(blob_service_client, output_container_url, output_blob_name, json.dumps(result, indent=4))
    print(f"Saved result to blob: {output_blob_name}")
